Schema evolution modes
- none
- fail on new col
- rescue
- add new cols (default)

In [0]:
%sql
use catalog ext_cat;

In [0]:
landing_zone =  '/Volumes/ext_cat/default/raw'
orders_data = landing_zone + '/ordershistory'
checkpoint_path =  landing_zone + '/orders_checkpoint'

**option('cloudFiles.schemaEvolutionMode','none')**

In [0]:
ordersdf= spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", checkpoint_path) \
  .option("cloudFiles.inferSchema", "true") \
  .option("cloudFiles.inferColumnTypes", 'true') \
  .option('cloudFiles.schemaEvolutionMode','none') \
  .load(orders_data) \
  

In [0]:
%sql
drop table if exists ext_cat.default.orders;

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

No rescued col because schemaEvolutionMode is 'none'

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status
1001,2026-08-25,201,Delivered
1002,2026-08-26,205,Shipped
1003,2026-08-27,202,Processing
1004,2026-08-28,208,Cancelled
1005,2026-08-30,201,Delivered
1006,2026-09-01,210,Shipped
1007,2026-09-02,204,Processing
1008,2026-09-03,207,Pending
1009,2026-09-05,203,Pending


add a new file with a new col and try

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

Job wont fail. New col completely ignored

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status
1001,2026-08-25,201,Delivered
1002,2026-08-26,205,Shipped
1003,2026-08-27,202,Processing
1004,2026-08-28,208,Cancelled
1005,2026-08-30,201,Delivered
1006,2026-09-01,210,Shipped
1007,2026-09-02,204,Processing
1008,2026-09-03,207,Pending
1009,2026-09-05,203,Pending
10101,2025-08-25,2101,Delivered


Add a file with data mismatch

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

Null gets loaded to cols with mismatched data

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status
1001,2026-08-25,201,Delivered
1002,2026-08-26,205,Shipped
1003,2026-08-27,202,Processing
1004,2026-08-28,208,Cancelled
1005,2026-08-30,201,Delivered
1006,2026-09-01,210,Shipped
1007,2026-09-02,204,Processing
1008,2026-09-03,207,Pending
1009,2026-09-05,203,Pending
10101,2025-08-25,2101,Delivered


**fail on new cols**

In [0]:
ordersdf= spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", checkpoint_path) \
  .option("cloudFiles.inferSchema", "true") \
  .option("cloudFiles.inferColumnTypes", 'true') \
  .option('cloudFiles.schemaEvolutionMode','failOnNewColumns') \
  .load(orders_data) \
  

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status,_rescued_data
1001,2026-08-25,201,Delivered,null
1002,2026-08-26,205,Shipped,null
1003,2026-08-27,202,Processing,null
1004,2026-08-28,208,Cancelled,null
1005,2026-08-30,201,Delivered,null
1006,2026-09-01,210,Shipped,null
1007,2026-09-02,204,Processing,null
1008,2026-09-03,207,Pending,null
1009,2026-09-05,203,Pending,null


upload file with new col- Job fails with error. Cannot be fixed by restart
`org.apache.spark.sql.catalyst.util.UnknownFieldException: [UNKNOWN_FIELD_EXCEPTION.NEW_FIELDS_IN_FILE] Encountered unknown fields during parsing: [new_col], which can be fixed by an automatic retry: false`

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

**rescue mode**

In [0]:
ordersdf= spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", checkpoint_path) \
  .option("cloudFiles.inferSchema", "true") \
  .option("cloudFiles.inferColumnTypes", 'true') \
  .option('cloudFiles.schemaEvolutionMode','rescue') \
  .load(orders_data) \
  

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status,_rescued_data
1001,2026-08-25,201,Delivered,null
1002,2026-08-26,205,Shipped,null
1003,2026-08-27,202,Processing,null
1004,2026-08-28,208,Cancelled,null
1005,2026-08-30,201,Delivered,null
1006,2026-09-01,210,Shipped,null
1007,2026-09-02,204,Processing,null
1008,2026-09-03,207,Pending,null
1009,2026-09-05,203,Pending,null


upload file with new col

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

Records with new col moved or data type mismatches to rescued data - no job failure - schema not changed

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status,_rescued_data
10101,2025-08-25,2101,Delivered,"{""new_col"":""1"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10102,2025-08-26,2105,Shipped,"{""new_col"":""1"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10103,2025-08-27,2102,Processing,"{""new_col"":""1"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10104,2025-08-28,2108,Cancelled,"{""new_col"":""1"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10105,2025-08-30,2101,Delivered,"{""new_col"":""1"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10106,2025-09-01,2110,Shipped,"{""new_col"":""1"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10107,2025-09-02,2104,Processing,"{""new_col"":""1"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10108,2025-09-03,2107,Pending,"{""new_col"":""1"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10109,2025-09-05,1203,Pending,"{""new_col"":""1"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
1001,2026-08-25,201,Delivered,null


**addnewColumns mode**

In [0]:
ordersdf= spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", checkpoint_path) \
  .option("cloudFiles.inferSchema", "true") \
  .option("cloudFiles.inferColumnTypes", 'true') \
  .option('cloudFiles.schemaEvolutionMode','addNewColumns') \
  .load(orders_data) \
  

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status,_rescued_data
1001,2026-08-25,201,Delivered,null
1002,2026-08-26,205,Shipped,null
1003,2026-08-27,202,Processing,null
1004,2026-08-28,208,Cancelled,null
1005,2026-08-30,201,Delivered,null
1006,2026-09-01,210,Shipped,null
1007,2026-09-02,204,Processing,null
1008,2026-09-03,207,Pending,null
1009,2026-09-05,203,Pending,null


add file with new col

`org.apache.spark.sql.catalyst.util.UnknownFieldException: [UNKNOWN_FIELD_EXCEPTION.NEW_FIELDS_IN_FILE] Encountered unknown fields during parsing: [new_col], which can be fixed by an automatic retry: true`

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

**New col added**

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status,_rescued_data,new_col
10101,2025-08-25,2101,Delivered,null,1
10102,2025-08-26,2105,Shipped,null,1
10103,2025-08-27,2102,Processing,null,1
10104,2025-08-28,2108,Cancelled,null,1
10105,2025-08-30,2101,Delivered,null,1
10106,2025-09-01,2110,Shipped,null,1
10107,2025-09-02,2104,Processing,null,1
10108,2025-09-03,2107,Pending,null,1
10109,2025-09-05,1203,Pending,null,1
1001,2026-08-25,201,Delivered,null,null


Data mismatch in file

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status,_rescued_data,new_col
10101,null,null,Delivered,"{""order_date"":""25-08-2025"",""customer_id"":""ABC"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",1
10102,null,2105,Shipped,"{""order_date"":""26-08-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",2
10103,null,2102,Processing,"{""order_date"":""27-08-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",3
10104,null,2108,Cancelled,"{""order_date"":""28-08-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",4
10105,null,2101,Delivered,"{""order_date"":""30-08-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",5
10106,null,2110,Shipped,"{""order_date"":""01-09-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",6
10107,null,2104,Processing,"{""order_date"":""02-09-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",7
10108,null,2107,Pending,"{""order_date"":""03-09-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",8
10109,null,1203,Pending,"{""order_date"":""05-09-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",1
10101,2025-08-25,2101,Delivered,null,1
